In [91]:
# Library

import os 
from getpass import getpass
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory, ConversationSummaryBufferMemory
from langchain_classic.chains import ConversationChain
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder, ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory, BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage, SystemMessage
from langchain_core.runnables import ConfigurableFieldSpec

load_dotenv()

True

In [35]:
# must enter API key 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# below should not be changed
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# you can change this as preferred
os.environ["LANGCHAIN_PROJECT"] = "damar-ananta-langsmith-openai"

In [36]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# For normal accurate responses
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

# ConversationBufferMemory

In [37]:
""" 
ConversationBufferMemory merupakan bentuk paling sederhana dari conversational memory. Konsepnya adalah menyimpan seluruh riwayat percakapan antara user dan AI.
"""

' \nConversationBufferMemory merupakan bentuk paling sederhana dari conversational memory. Konsepnya adalah menyimpan seluruh riwayat percakapan antara user dan AI.\n'

In [38]:
# 1. Membuat memory
memory = ConversationBufferMemory(return_messages=True) #return_messages=True: mengembalikan riwayat percakapan dalam bentuk list objek message, seperti HumanMessage dan AIMessage.

In [39]:
# 2. Menambahkan percakapan ke memory 
memory.save_context(
    {"input": "Hi, my name is James"}, # input -> Menyimpan pesan dari user
    {"output": "Hey James, what's up " } # output -> Menyimpan jawaban dari AI
)

In [40]:
memory.save_context(
    {"input": "I'm learning LangChain"},
    {"output": "That's great!"}
)

memory.save_context(
    {"input": "What is conversational memory?"},
    {"output": "It stores conversation history."}
)

In [41]:
# 3. Melihat isi memory 
memory.load_memory_variables({}) 

# Perhatikan key history. Key tersebut berisi seluruh percakapan user dan AI yang sudah disimpan.
# Mengapa parameternya {} kosong? Karena ConversationBufferMemory tidak membutuhkan input tambahan untuk mengambil riwayat percakapan

{'history': [HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey James, what's up ", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I'm learning LangChain", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's great!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is conversational memory?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='It stores conversation history.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [42]:
# 4. Menghubungkan Memory dengan LLM

chain = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True # menampilkan proses internal chain, termasuk prompt yang dikirimkan ke LLM.
)

In [43]:
chain.invoke({
    "input": "what is my name again?"
})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}), AIMessage(content="Hey James, what's up ", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I'm learning LangChain", additional_kwargs={}, response_metadata={}), AIMessage(content="That's great!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is conversational memory?', additional_kwargs={}, response_metadata={}), AIMessage(content='It stores conversation history.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Human: what is

{'input': 'what is my name again?',
 'history': [HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hey James, what's up ", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I'm learning LangChain", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's great!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is conversational memory?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='It stores conversation history.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='what is my name again?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Your name is James!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'response': 'Your name is James!'}

## ConversationBufferMemory with RunnableWithMessageHistory

In [44]:
""" 
ConversationChain dan ConversationBufferMemory merupakan API lama yang sudah deprecated. Sebagai alternatif, LangChain memperkenalkan RunnableWithMessageHistory.

    ConversationChain	                        RunnableWithMessageHistory
Menggunakan sistem Chain lama	            Menggunakan sistem Runnable
Memory dipasang melalui parameter memory	History dikelola melalui fungsi session history
Lebih terikat pada abstraksi Chain lama	    Lebih fleksibel dengan LCEL

Hal penting yang perlu kamu pahami: RunnableWithMessageHistory bukan sekadar pengganti nama ConversationBufferMemory.

Kita perlu menyiapkan sendiri:
1. Prompt template.
2. Pipeline antara prompt dan LLM.
3. Penyimpanan message history.
4. Fungsi untuk mengambil history berdasarkan session ID.
"""

' \nConversationChain dan ConversationBufferMemory merupakan API lama yang sudah deprecated. Sebagai alternatif, LangChain memperkenalkan RunnableWithMessageHistory.\n\n    ConversationChain\t                        RunnableWithMessageHistory\nMenggunakan sistem Chain lama\t            Menggunakan sistem Runnable\nMemory dipasang melalui parameter memory\tHistory dikelola melalui fungsi session history\nLebih terikat pada abstraksi Chain lama\t    Lebih fleksibel dengan LCEL\n\nHal penting yang perlu kamu pahami: RunnableWithMessageHistory bukan sekadar pengganti nama ConversationBufferMemory.\n\nKita perlu menyiapkan sendiri:\n1. Prompt template.\n2. Pipeline antara prompt dan LLM.\n3. Penyimpanan message history.\n4. Fungsi untuk mengambil history berdasarkan session ID.\n'

In [45]:
# Mmebuat ChatPromptTemplate

system_prompt = "You are a helpful assistant called Zeta"

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt), # Digunakan untuk mendefinisikan instruksi utama bagi LLM. Artinya, kita menginstruksikan LLM untuk berperan sebagai asisten bernama Zeta.
    MessagesPlaceholder(variable_name="history"), # MessagesPlaceholder menyediakan tempat untuk memasukkan kumpulan percakapan sebelumnya. Perhatikan variable_name="history". Nama history nantinya akan diisi dengan kumpulan message dari penyimpanan chat history.
    HumanMessagePromptTemplate.from_template("{query}") # Placeholder {query} nantinya akan diisi dengan pertanyaan aktual user.
])

In [46]:
# Menggabungkan prompt dengan LLM 
pipeline = prompt_template | llm # Operator | disebut pipe operator. Fungsinya menghubungkan satu Runnable dengan Runnable

In [47]:
# Membuat penyimpanan chat history
chat_map = {} #Ini adalah dictionary Python yang digunakan untuk menyimpan history berdasarkan session ID.

def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in chat_map:
        chat_map[session_id] = InMemoryChatMessageHistory() ##Class ini digunakan untuk menyimpan kumpulan pesan percakapan di dalam memory (RAM).
    """ 
    Artinya:
        - Jika session ID belum tersedia, buat penyimpanan history baru.
        - Jika session ID sudah tersedia, gunakan history yang sudah ada.       
    """
    return chat_map[session_id]


In [48]:
# Membungkus pipeline dengan RunnableWithMessageHistory

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history, #Fungsi untuk mengambil history berdasarkan session ID
    input_messages_key="query", #Menentukan key untuk pesan terbaru dari user. harus sesuai dengan nama placeholder yang sudah kita definisikan pada ChatPromptTemplate.
    history_messages_key="history" #Menentukan key untuk memasukkan history ke prompt. harus sesuai dengan nama placeholder yang sudah kita definisikan pada ChatPromptTemplate.
)

/Users/admin/Library/Python/3.11/lib/python/site-packages/IPython/core/interactiveshell.py:3549: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [49]:
# Menjalankan RunnableWithMessageHistory

pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"session_id": "id_123"} # id percakapan
)

AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20c76-004c-7f21-94f4-373864a06a9d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [50]:
pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={"session_id": "id_123"}
)

AIMessage(content='Your name is James. How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 49, 'total_tokens': 61, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9400314673', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20c76-04c2-7030-9f22-e6a079a54ed6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 12, 'total_tokens': 61, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# ConversationBufferWindowMemory

In [51]:
""" 
Pada materi pertama, kita sudah mempelajari ConversationBufferMemory, yaitu memory yang menyimpan seluruh riwayat percakapan.

Masalahnya, semakin banyak percakapan, semakin banyak token yang dikirimkan ke LLM.

ConversationBufferWindowMemory hadir untuk mengatasi masalah tersebut dengan membatasi jumlah pesan yang dimasukkan ke dalam konteks LLM.

Notebook menjelaskan tiga alasan mengapa kita perlu membatasi riwayat percakapan:
1. Mengurangi penggunaan token: semakin banyak pesan, semakin banyak token yang dikirimkan. Ini meningkatkan biaya API dan latency.
2. Menjaga kualitas jawaban: konteks yang terlalu panjang bisa membuat LLM lebih sulit fokus pada informasi relevan.
3. Menghindari context window limit: setiap LLM memiliki batas jumlah token yang dapat diproses dalam satu konteks.

Misalnya:
- k=1: menyimpan 1 interaksi terakhir.
- k=2: menyimpan 2 interaksi terakhir.
- k=3: menyimpan 3 interaksi terakhir.
Satu interaksi terdiri dari satu pesan Human dan satu pesan AI.
"""

' \nPada materi pertama, kita sudah mempelajari ConversationBufferMemory, yaitu memory yang menyimpan seluruh riwayat percakapan.\n\nMasalahnya, semakin banyak percakapan, semakin banyak token yang dikirimkan ke LLM.\n\nConversationBufferWindowMemory hadir untuk mengatasi masalah tersebut dengan membatasi jumlah pesan yang dimasukkan ke dalam konteks LLM.\n\nNotebook menjelaskan tiga alasan mengapa kita perlu membatasi riwayat percakapan:\n1. Mengurangi penggunaan token: semakin banyak pesan, semakin banyak token yang dikirimkan. Ini meningkatkan biaya API dan latency.\n2. Menjaga kualitas jawaban: konteks yang terlalu panjang bisa membuat LLM lebih sulit fokus pada informasi relevan.\n3. Menghindari context window limit: setiap LLM memiliki batas jumlah token yang dapat diproses dalam satu konteks.\n\nMisalnya:\n- k=1: menyimpan 1 interaksi terakhir.\n- k=2: menyimpan 2 interaksi terakhir.\n- k=3: menyimpan 3 interaksi terakhir.\nSatu interaksi terdiri dari satu pesan Human dan satu p

In [ ]:
# Membuat memori

memory = ConversationBufferWindowMemory(
    k=4,
    return_messages=True #Mengembalikan history dalam bentuk objek message seperti HumanMessage dan AIMessage.
)

In [53]:
# Menambahkan percakapan
memory.chat_memory.add_user_message("Hi, my name is James")
memory.chat_memory.add_ai_message("Hey James, what's up? I'm an AI model called Zeta.")
memory.chat_memory.add_user_message("I'm researching the different types of conversational memory.")
memory.chat_memory.add_ai_message("That's interesting, what are some examples?")
memory.chat_memory.add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
memory.chat_memory.add_ai_message("That's interesting, what's the difference?")
memory.chat_memory.add_user_message("Buffer memory just stores the entire conversation, right?")
memory.chat_memory.add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
memory.chat_memory.add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
memory.chat_memory.add_ai_message("Very cool!")

In [54]:
# Mengambil history

memory.load_memory_variables({}) #Jadi, percakapan pertama tidak muncul karena sudah berada di luar batas window.

{'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwa

In [55]:
# Menghubungkan Window Memory dengan ConversationChain

chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

In [56]:
chain.invoke({
    "input": "what is my name again?"
})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}), AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs

{'input': 'what is my name again?',
 'history': [HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what are some examples?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="That's interesting, what's the difference?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Buffer window memory stores the last k messages,

## ConversationBufferWindowMemory with RunnableWithMessageHistory

In [ ]:
# Membuat custom class BufferWindowMessageHistory karena di materi ini digunakan untuk menyimpan seluruh percakapan dan sekarang ada tambahan mekanisme untuk membatasai jumlah pesan

class BufferWindowMessageHistory(BaseChatMessageHistory # menyediakan interface dasar untuk pengelolaan riwayat percakapan.
                                 , BaseModel # berasal dari Pydantic, digunakan untuk mengelola dan memvalidasi data pada model.
                                 ):
    messages: list[BaseMessage] = Field(default_factory=list) # Digunakan untuk menyimpan kumpulan pesan percakapan, default_factory=list -> Artinya, setiap objek baru akan mendapatkan list kosong sebagai nilai awal. Mengapa tidak langsung menggunakan messages = []? Karena default_factory memastikan setiap instance memiliki list masing-masing, bukan berbagi list yang sama.
    k: int = Field(default_factory=int) #Digunakan untuk menyimpan jumlah maksimal pesan yang ingin dipertahankan.

    def __init__(self, k: int):
        super().__init__(k=4)
        print(f"Initializing BufferWindowMessagesHistory with k={k}")
    
    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.messages.extend(messages) # Digunakan untuk menambahkan message baru ke dalam list yang sudah ada.
        self.messages = self.messages[-self.k:] #Tujuannya mengambil hanya k elemen terakhir dari list.

    def clear(self) -> None: # Digunakan untuk menghapus seluruh riwayat percakapan.
        self.messages = []

In [ ]:
# Memebuat fungsi get_chat_history()

chat_map = {}

def get_chat_history(session_id: str, k: int = 4,) -> BufferWindowMessageHistory:
    print(
        f"get_chat_history called with session_id={session_id} and k={k}"
    )

    if session_id not in chat_map: # Jika session belum tersedia, buat objek history baru dengan batas k yang diberikan.
        chat_map[session_id] = BufferWindowMessageHistory(k=k)

        # Jika session sudah tersedia, gunakan objek history yang sudah ada.
        # Jadi, setiap session memiliki window memory masing-masing.

    return chat_map[session_id]


In [ ]:

# Membungkus pipeline dengan RunnableWithMessageHistory

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec( # digunakan untuk mendefinisikan parameter konfigurasi yang dibutuhkan oleh history factory.
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)

/Users/admin/Library/Python/3.11/lib/python/site-packages/IPython/core/interactiveshell.py:3549: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [63]:
# Menjalankan pipeline dengan parameter K

pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={
        "configurable": {
            "session_id": "id_k4",
            "k": 4
        }
    }
)

get_chat_history called with session_id=id_k4 and k=4
Initializing BufferWindowMessagesHistory with k=4


AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20ca1-b14b-7cc0-9bb4-4370d9035d3e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [64]:
# Momodifikasi history secara manual

chat_map["id_k4"].clear()

In [67]:
# manually insert history
chat_map["id_k4"].add_user_message("Hi, my name is James")
chat_map["id_k4"].add_ai_message("I'm an AI model called Zeta.")
chat_map["id_k4"].add_user_message("I'm researching the different types of conversational memory.")
chat_map["id_k4"].add_ai_message("That's interesting, what are some examples?")
chat_map["id_k4"].add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
chat_map["id_k4"].add_ai_message("That's interesting, what's the difference?")
chat_map["id_k4"].add_user_message("Buffer memory just stores the entire conversation, right?")
chat_map["id_k4"].add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
chat_map["id_k4"].add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
chat_map["id_k4"].add_ai_message("Very cool!")

In [ ]:
# Memeriksa isi history
chat_map['id_k4'].messages

[HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [69]:
pipeline_with_history.invoke(
    {"query": "what is my name again?"},
    config={"configurable": {"session_id": "id_k4", "k": 4}}
)

get_chat_history called with session_id=id_k4 and k=4


AIMessage(content="I'm sorry, but I don't have access to personal information like your name unless you tell me. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 78, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_48eea167bf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20ca4-0146-7573-b75c-8038626821ba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 26, 'total_tokens': 104, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [70]:
pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"session_id": "id_k14", "k": 14}
)

get_chat_history called with session_id=id_k14 and k=14
Initializing BufferWindowMessagesHistory with k=14


AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20ca4-4f81-7671-a846-8cc9205016aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [71]:
chat_map["id_k14"].add_user_message("I'm researching the different types of conversational memory.")
chat_map["id_k14"].add_ai_message("That's interesting, what are some examples?")
chat_map["id_k14"].add_user_message("I've been looking at ConversationBufferMemory and ConversationBufferWindowMemory.")
chat_map["id_k14"].add_ai_message("That's interesting, what's the difference?")
chat_map["id_k14"].add_user_message("Buffer memory just stores the entire conversation, right?")
chat_map["id_k14"].add_ai_message("That makes sense, what about ConversationBufferWindowMemory?")
chat_map["id_k14"].add_user_message("Buffer window memory stores the last k messages, dropping the rest.")
chat_map["id_k14"].add_ai_message("Very cool!")

In [72]:
chat_map["id_k14"].messages

[HumanMessage(content='Buffer memory just stores the entire conversation, right?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='That makes sense, what about ConversationBufferWindowMemory?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [73]:
pipeline_with_history.invoke(
    {"query": "what is my name again?"},
    config={"session_id": "id_k14", "k": 14}
)

get_chat_history called with session_id=id_k14 and k=14


AIMessage(content="I'm sorry, but I don't have access to personal information about users unless you've shared it in this conversation. If you'd like me to remember your name for this chat, please let me know!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 78, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_48eea167bf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20ca5-05a1-7f93-a295-9fbdf3f8d1e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 38, 'total_tokens': 116, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [74]:
chat_map["id_k14"].messages

[HumanMessage(content='Buffer window memory stores the last k messages, dropping the rest.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Very cool!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is my name again?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm sorry, but I don't have access to personal information about users unless you've shared it in this conversation. If you'd like me to remember your name for this chat, please let me know!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 78, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_48eea167bf', 'finish_reason':

# ConversationSummaryMemory

In [ ]:
""" 
ConversationSummaryMemory merupakan jenis conversational memory yang menyimpan ringkasan dari percakapan sebelumnya, bukan seluruh pesan percakapan.

Tujuannya adalah mengurangi jumlah token yang digunakan tanpa kehilangan seluruh konteks percakapan.

ConversationSummaryMemory menghemat penggunaan konteks dengan mengorbankan sebagian detail percakapan.
"""

In [ ]:
# Membuat memory
memory = ConversationSummaryMemory(
    llm=llm
)

/var/folders/sd/7_dvq1bs4x12529lgl1190_w0000gn/T/ipykernel_38936/1436668449.py:1: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


In [77]:
# Menghubungkan memmory dengan ConversationChain

chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

In [78]:
chain.invoke({
    "input": "hello there my name is James"
})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: hello there my name is James
AI:

> Finished chain.


{'input': 'hello there my name is James',
 'history': '',
 'response': "Hello, James! It's great to meet you! I'm here to chat and help with anything you might need. How's your day going so far?"}

In [79]:
chain.invoke({
    "input": "I am researching the different types of conversational memory"
})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
The human introduces himself as James. The AI greets James and expresses its readiness to chat and assist, inquiring about how his day is going.
Human: I am researching the different types of conversational memory
AI:

> Finished chain.


{'input': 'I am researching the different types of conversational memory',
 'history': 'The human introduces himself as James. The AI greets James and expresses its readiness to chat and assist, inquiring about how his day is going.',
 'response': 'That sounds like an interesting topic, James! Conversational memory refers to how systems, like chatbots or virtual assistants, remember information from past interactions to improve future conversations. There are a few different types of conversational memory:\n\n1. **Short-term Memory**: This type allows the AI to remember details from the current conversation. For example, if you mention your favorite color, the AI can refer back to that within the same chat session.\n\n2. **Long-term Memory**: This involves storing information across multiple interactions. For instance, if you tell the AI your name or preferences, it can recall that information in future conversations, creating a more personalized experience.\n\n3. **Contextual Memory**

In [ ]:
# Menguji apakah LLM masih mengingat nama user
chain.invoke({
    "input": "what  is my name again?"
})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
The human introduces himself as James, and the AI greets him, asking about his day. James mentions he is researching different types of conversational memory. The AI explains that conversational memory includes several types: short-term memory for details within a single conversation, long-term memory for information across multiple interactions, contextual memory for maintaining dialogue coherence, dynamic memory for adapting to new information, and episodic memory for recalling specific past events. The AI then asks if James is focusing on a specific type of conversational memory or exploring them all.
Human: what  is my name again?
AI:

> Finished chain.


{'input': 'what  is my name again?',
 'history': 'The human introduces himself as James, and the AI greets him, asking about his day. James mentions he is researching different types of conversational memory. The AI explains that conversational memory includes several types: short-term memory for details within a single conversation, long-term memory for information across multiple interactions, contextual memory for maintaining dialogue coherence, dynamic memory for adapting to new information, and episodic memory for recalling specific past events. The AI then asks if James is focusing on a specific type of conversational memory or exploring them all.',
 'response': "Your name is James! How's your research going? Are you focusing on a specific type of conversational memory?"}

## ConversationSummaryMemory with RunnableWithMessageHistory

In [ ]:
class ConversationSummaryMessageHistory(
    BaseChatMessageHistory # menyediakan interface untuk mengelola chat history.
    , BaseModel #mengelola atribut dan validasi data menggunakan Pydantic.
    ):
    messages: list[BaseMessage] = Field( 
        default_factory=list
    ) # Menyimpan message history, dalam implementasi ini berupa satu SystemMessage berisi summary

    llm: ChatOpenAI = Field(
        default_factory=ChatOpenAI
    ) # Model yang digunakan untuk membuat summary

    def __init__(self, llm: ChatOpenAI):
        super().__init__(llm=llm)

    def add_messages(
        self, messages: list[BaseMessage]
    ) -> None:

        self.messages.extend(messages)

        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template( # Memberikan instruksi kepada LLM agar membuat ringkasan baru dengan mempertahankan informasi relevan sebanyak mungkin.
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensuring to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template( 
                "Existing conversation summary:\n{existing_summary}\n\n" # {existing_summary}: ringkasan sebelumnya
                "New messages:\n{messages}" # {messages}: pesan terbaru.
            )
        ])

        new_summary = self.llm.invoke( # Memanggil LLM untuk membuat summary
            summary_prompt.format_messages(
                existing_summary=self.messages.content,
                messages=[x.content for x in messages]
            )
        )

        self.messages = [ # Mengganti history dengan summary terbaru
            SystemMessage(content=new_summary.content)
        ]

    def clear(self) -> None:
        self.messages = []

In [83]:
# Membuat fungsi get_chat_history()

chat_map = {}

def get_chat_history(
    session_id: str,
    llm: ChatOpenAI
) -> ConversationSummaryMessageHistory:

    if session_id not in chat_map:
        chat_map[session_id] = ConversationSummaryMessageHistory(
            llm=llm
        )

    return chat_map[session_id]

In [84]:
# Menghubungkan dengan RunnableWithMessageHistory

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=ChatOpenAI,
            name="LLM",
            description="The LLM to use for the conversation summary",
            default=llm,
        )
    ]
)

/Users/admin/Library/Python/3.11/lib/python/site-packages/IPython/core/interactiveshell.py:3549: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [85]:
# Menjalankan pipeline

pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={
        "session_id": "id_123",
        "llm": llm
    }
)

Error in RootListenersTracer.on_chain_end callback: AttributeError("'list' object has no attribute 'content'")


AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20cb8-f94a-7302-836b-52d7e9316e3d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [86]:
# Melihat hasil summary
chat_map["id_123"].messages

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20cb8-f94a-7302-836b-52d7e9316e3d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

In [87]:
# Melanjutkan percakapan
pipeline_with_history.invoke(
    {
        "query": "I'm researching the different types of conversational memory."
    },
    config={
        "session_id": "id_123",
        "llm": llm
    }
)

Error in RootListenersTracer.on_chain_end callback: AttributeError("'list' object has no attribute 'content'")


AIMessage(content='That sounds interesting! Conversational memory can refer to various concepts depending on the context, such as in human communication, artificial intelligence, or even in the study of social interactions. Here are a few types of conversational memory you might consider:\n\n1. **Short-term Memory**: This refers to the ability to hold a small amount of information in an active state for a brief period. In conversations, this allows individuals to remember what was just said to respond appropriately.\n\n2. **Long-term Memory**: This involves the storage of information over extended periods. In conversations, long-term memory helps individuals recall past interactions, shared experiences, or important details about the other person.\n\n3. **Contextual Memory**: This type of memory involves remembering the context in which a conversation took place, including the setting, emotions, and social dynamics. It helps individuals understand the nuances of a conversation.\n\n4. *

In [88]:
# Menguji beberapa percakapan

for msg in [
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    pipeline_with_history.invoke(
        {"query": msg},
        config={
            "session_id": "id_123",
            "llm": llm
        }
    )

Error in RootListenersTracer.on_chain_end callback: AttributeError("'list' object has no attribute 'content'")
Error in RootListenersTracer.on_chain_end callback: AttributeError("'list' object has no attribute 'content'")
Error in RootListenersTracer.on_chain_end callback: AttributeError("'list' object has no attribute 'content'")


In [89]:
chat_map["id_123"].messages

[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20cb8-f94a-7302-836b-52d7e9316e3d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content="I'm researching the different types of conversational memory.", additional_kwargs

In [90]:
pipeline_with_history.invoke(
    {"query": "What is my name again?"},
    config={
        "session_id": "id_123",
        "llm": llm
    }
)

Error in RootListenersTracer.on_chain_end callback: AttributeError("'list' object has no attribute 'content'")


AIMessage(content='Your name is James! How can I assist you further today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 1697, 'total_tokens': 1710, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1664}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe5dc95ec4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20cbb-7cea-7ef2-849d-9db2d6efdd9c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1697, 'output_tokens': 13, 'total_tokens': 1710, 'input_token_details': {'audio': 0, 'cache_read': 1664}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# ConversationSummaryBufferMemory

In [ ]:
""" 
ConversationSummaryBufferMemory. Ini merupakan materi terakhir dari empat jenis conversational memory pada notebook.

Materi ini merupakan gabungan dari dua konsep yang sudah kita pelajari:
- ConversationBufferWindowMemory: mempertahankan sejumlah pesan terakhir.
- ConversationSummaryMemory: membuat ringkasan dari percakapan sebelumnya.
Jadi, pendekatan ini mencoba mendapatkan keuntungan dari keduanya: mempertahankan percakapan terbaru secara lengkap, tetapi tetap menyimpan konteks percakapan lama melalui summary.

Cara kerjanya:
1. Menyimpan percakapan terbaru dalam bentuk message asli.
2. Ketika jumlah token melewati batas yang ditentukan, pesan-pesan paling lama dikeluarkan dari buffer.
3. Pesan lama tersebut diringkas menggunakan LLM.
4. Summary lama diperbarui dengan informasi dari pesan yang dikeluarkan.
5. Summary disimpan bersama pesan-pesan terbaru.
"""

In [ ]:
# Membuat memory 

memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=300, # Ini merupakan parameter yang menentukan batas token untuk buffer percakapan.
    return_messages=True
)

/var/folders/sd/7_dvq1bs4x12529lgl1190_w0000gn/T/ipykernel_38936/1045153024.py:3: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryBufferMemory(


In [ ]:
# Menghubungkan dengan ConversationChain
chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True # memungkinkan kita melihat bagaimana prompt dan history berubah ketika percakapan berlangsung.
)

In [94]:
# Menjalankan percakapan pertama

chain.invoke({
    "input": "Hi, my name is James"
})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[]
Human: Hi, my name is James
AI:

> Finished chain.


{'input': 'Hi, my name is James',
 'history': [HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello, James! It's great to meet you. How's your day going so far?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'response': "Hello, James! It's great to meet you. How's your day going so far?"}

In [95]:
# Menambahkan beberapa percakapan

for msg in [
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]:
    chain.invoke({"input": msg})



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello, James! It's great to meet you. How's your day going so far?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Human: I'm researching the different types of conversational memory.
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:


In [ ]:
""" 
- SystemMessage berisi summary percakapan lama.
- HumanMessage dan AIMessage berisi percakapan yang masih berada dalam buffer.
"""

## ConversationSummaryBufferMemory with RunnableWithMessageHistory

In [ ]:
class ConversationSummaryBufferMessageHistory(
    BaseChatMessageHistory # menyediakan interface untuk mengelola chat history.
    , BaseModel # mengelola atribut menggunakan Pydantic.
    ):
    messages: list[BaseMessage] = Field(default_factory=list) # Menyimpan summary dan pesan terbaru
    llm: ChatOpenAI = Field(default_factory=ChatOpenAI) # Model untuk menghasilkan summary
    k: int = Field(default_factory=int) # Batas jumlah pesan terbaru yang dipertahanka

    def __init__(self, llm: ChatOpenAI, k: int):
        super().__init__(llm=llm, k=k)

    def add_messages(self, messages: list[BaseMessage]) -> None: # Method ini akan dipanggil ketika ada pesan baru yang perlu ditambahkan ke history.
        """Add messages to the history, removing any messages beyond
        the last `k` messages and summarizing the messages that we
        drop.
        """
        existing_summary: SystemMessage | None = None # Menyimpan summary yang sudah ada
        old_messages: list[BaseMessage] | None = None # Menyimpan pesan lama yang akan diringkas
        # Tanda | None berarti variabel tersebut boleh berisi objek sesuai tipe yang ditentukan atau None. Pada awalnya, keduanya belum memiliki nilai.

        # Memeriksa apakah sudah ada summary
        if len(self.messages) > 0 and isinstance(self.messages[0], SystemMessage): # Memastikan bahwa history tidak kosong dan Memeriksa apakah message pertama merupakan SystemMessage.
            print(">> Found existing summary")
            existing_summary = self.messages.pop(0) #Jadi, jika sebelumnya terdapat summary, summary tersebut dipindahkan ke variabel existing_summary. Mengapa dihapus dari list sementara? Agar summary lama bisa digunakan dalam proses pembuatan summary baru, kemudian nantinya diganti dengan summary terbaru.
        
        # Menambahkan pesan baru
        self.messages.extend(messages)

        # Memeriksa jumlah pesan
        if len(self.messages) > self.k:
            print(
                f">> Found {len(self.messages)} messages, dropping "
                f"oldest {len(self.messages) - self.k} messages.")
            # Memisahkan pesan lama dan pesan terbaru
            old_messages = self.messages[:-self.k] # old_messages: pesan lama yang akan diringkas.
            self.messages = self.messages[-self.k:] # self.messages: pesan terbaru yang tetap disimpan secara lengkap.
        
        # Memeriksa apakah ada pesan lama
        if old_messages is None:
            print(">> No old messages to update summary with")
            # if we have no old_messages, we have nothing to update in summary
            return
        
        # Setelah pesan lama berhasil dipisahkan, notebook membuat prompt untuk menghasilkan summary baru.
        summary_prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "Given the existing conversation summary and the new messages, "
                "generate a new summary of the conversation. Ensuring to maintain "
                "as much relevant information as possible."
            ),
            HumanMessagePromptTemplate.from_template(
                "Existing conversation summary:\n{existing_summary}\n\n"
                "New messages:\n{old_messages}"
            )
        ])
        # Memanggil LLM untuk membuat summary
        new_summary = self.llm.invoke(
            summary_prompt.format_messages(
                existing_summary=existing_summary, # existing_summary diisi dengan summary sebelumnya.
                old_messages=old_messages # old_messages diisi dengan pesan lama yang dikeluarkan dari buffer.
            )
        )
        print(f">> New summary: {new_summary.content}")

        # Menambahkan summary ke history
        self.messages = [SystemMessage(content=new_summary.content)] + self.messages

    def clear(self) -> None:
        """Clear the history."""
        self.messages = []

In [ ]:

chat_map = {}
def get_chat_history(session_id: str, llm: ChatOpenAI, k: int) -> ConversationSummaryBufferMessageHistory: # Simbol -> digunakan untuk memberikan return type annotation atau petunjuk tipe data yang akan dikembalikan oleh function. memberikan informasi bahwa function get_chat_history() diharapkan mengembalikan objek bertipe ConversationSummaryBufferMessageHistory.
    if session_id not in chat_map:
        # if session ID doesn't exist, create a new chat history
        chat_map[session_id] = ConversationSummaryBufferMessageHistory(llm=llm, k=k)
    # return the chat history
    return chat_map[session_id]

In [99]:

pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="The session ID to use for the chat history",
            default="id_default",
        ),
        ConfigurableFieldSpec(
            id="llm",
            annotation=ChatOpenAI,
            name="LLM",
            description="The LLM to use for the conversation summary",
            default=llm,
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="k",
            description="The number of messages to keep in the history",
            default=4,
        )
    ]
)

/Users/admin/Library/Python/3.11/lib/python/site-packages/IPython/core/interactiveshell.py:3549: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
# Menjalankan pipeline
pipeline_with_history.invoke(
    {"query": "Hi, my name is James"},
    config={"session_id": "id_123", "llm": llm, "k": 4}
)
chat_map["id_123"].messages

>> No old messages to update summary with


[HumanMessage(content='Hi, my name is James', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi James! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 25, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_96719e0fde', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a20cc9-3903-7e91-b795-778daddaa60c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 10, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

In [101]:
for i, msg in enumerate([
    "I'm researching the different types of conversational memory.",
    "I have been looking at ConversationBufferMemory and ConversationBufferWindowMemory.",
    "Buffer memory just stores the entire conversation",
    "Buffer window memory stores the last k messages, dropping the rest."
]):

    print(f"---\nMessage {i+1}\n---\n")

    pipeline_with_history.invoke(
        {"query": msg},
        config={
            "session_id": "id_123",
            "llm": llm,
            "k": 4
        }
    )

---
Message 1
---

>> No old messages to update summary with
---
Message 2
---

>> Found 6 messages, dropping oldest 2 messages.
>> New summary: James introduced himself in the conversation, and the AI responded by asking how it could assist him today.
---
Message 3
---

>> Found existing summary
>> Found 6 messages, dropping oldest 2 messages.
>> New summary: In the conversation, James introduced himself, and the AI inquired how it could assist him. James mentioned that he is researching different types of conversational memory. The AI responded by outlining several types of conversational memory, including:

1. **Short-term Memory**: Retaining information for brief periods to respond appropriately.
2. **Long-term Memory**: Storing information over longer periods, including past interactions and personal experiences.
3. **Contextual Memory**: Recalling the context of information learned or discussed.
4. **Semantic Memory**: Memory of facts and general knowledge for discussing shared t

In [102]:
chat_map["id_123"].messages

[SystemMessage(content='In the conversation, James introduced himself and mentioned that he is researching different types of conversational memory. The AI provided an overview of several types of conversational memory, including short-term, long-term, contextual, semantic, episodic, conversational memory in AI, and social memory. The AI then inquired if James was focusing on a specific area or needed more detailed information.\n\nJames subsequently mentioned that he has been looking at **ConversationBufferMemory** and **ConversationBufferWindowMemory**. The AI explained both concepts in detail:\n\n1. **ConversationBufferMemory**: \n   - Stores the entire conversation history, allowing the AI to maintain context and continuity.\n   - Ideal for scenarios requiring full context, such as customer support.\n   - Advantages include nuanced responses and better understanding of user intent, while disadvantages involve potential performance issues with long conversations and complex memory ma

In [ ]:
""" 
1. Apa itu Buffer?
Secara umum, buffer adalah tempat penyimpanan sementara data sebelum data tersebut diproses, digunakan, atau dipindahkan lebih lanjut.

2. Apa itu Window Memory?
- Window berarti jendela atau batas tertentu.
- Dalam konteks conversational memory, Window Memory adalah mekanisme yang membatasi berapa banyak pesan percakapan terakhir yang dipertahankan untuk digunakan sebagai konteks LLM.
"""